# 01 — Data Exploration

This notebook explores the DEAP dataset: EEG signals, scalograms, label distributions,
and HRV feature extraction.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

# Synthetic data for demo (replace with real dataset paths)
np.random.seed(42)
print('Libraries loaded successfully.')

## 1. Synthetic EEG Signal Visualization

In [ ]:
sfreq = 128
duration = 4  # seconds
t = np.arange(0, duration, 1/sfreq)

# Synthetic EEG: mixture of alpha (10 Hz) and beta (20 Hz)
eeg_ch1 = (np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*20*t)
           + 0.2*np.random.randn(len(t)))
eeg_ch2 = (0.8*np.sin(2*np.pi*8*t) + 0.3*np.sin(2*np.pi*15*t)
           + 0.2*np.random.randn(len(t)))

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(t, eeg_ch1, lw=0.8, color='#58a6ff')
axes[0].set_ylabel('Channel 1 (µV)', fontsize=10)
axes[0].set_title('Synthetic EEG Signal', fontsize=12)
axes[1].plot(t, eeg_ch2, lw=0.8, color='#3fb950')
axes[1].set_ylabel('Channel 2 (µV)', fontsize=10)
axes[1].set_xlabel('Time (s)', fontsize=10)
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 2. CWT Scalogram

In [ ]:
from preprocessing.wavelet_transform import compute_scalogram

scalogram = compute_scalogram(eeg_ch1.astype(np.float32), sfreq=sfreq, n_freqs=32)
print(f'Scalogram shape: {scalogram.shape}')  # (32, 512)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(
    np.log1p(scalogram),
    aspect='auto', origin='lower', cmap='viridis',
    extent=[0, duration, 0.5, 45]
)
ax.set_xlabel('Time (s)', fontsize=10)
ax.set_ylabel('Frequency (Hz)', fontsize=10)
ax.set_title('CWT Morlet Scalogram (log power)', fontsize=12)
plt.colorbar(im, ax=ax, label='log(1 + power)')
ax.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 3. Label Distribution (Simulated DEAP)

In [ ]:
from data.dataset_loaders import _deap_av_to_cognitive

# Simulate DEAP arousal/valence labels
rng = np.random.default_rng(0)
n_trials = 1280  # 32 subjects × 40 trials
arousal = rng.uniform(1, 9, n_trials)
valence = rng.uniform(1, 9, n_trials)
labels = [_deap_av_to_cognitive(a, v) for a, v in zip(arousal, valence)]
class_names = ['Underload', 'Optimal', 'Overload', 'Fatigue']
counts = Counter(labels)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#58a6ff', '#3fb950', '#f78166', '#d29922']
bars = ax.bar([class_names[i] for i in range(4)],
              [counts.get(i, 0) for i in range(4)],
              color=colors, edgecolor='#30363d', linewidth=0.5)
for bar, count in zip(bars, [counts.get(i, 0) for i in range(4)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', va='bottom', color='#8b949e', fontsize=10)
ax.set_title('Cognitive State Label Distribution (Simulated DEAP)', fontsize=12)
ax.set_ylabel('Count', fontsize=10)
ax.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()
print('Label distribution:', {class_names[k]: v for k, v in counts.items()})

## 4. HRV Feature Extraction

In [ ]:
from preprocessing.hrv_preprocessor import HRVFeatureExtractor

# Simulate PPG signal (60 bpm)
t_ppg = np.arange(0, 60, 1/128)
ppg = np.sin(2*np.pi*1.0*t_ppg) + 0.1*np.random.randn(len(t_ppg))

extractor = HRVFeatureExtractor(sfreq=128)
rr = extractor.extract_rr_intervals(ppg.astype(np.float32))
features = extractor.extract_features(ppg.astype(np.float32))

print('RR intervals (ms):', rr[:10].round(1))
feat_names = ['mean_rr', 'sdnn', 'rmssd', 'pnn50', 'mean_hr',
               'lf_power', 'hf_power', 'vlf_power', 'lf_hf_ratio', 'total_power']
for name, val in zip(feat_names, features):
    print(f'  {name:15s}: {val:.4f}')